# 06. 인공 신경망 (Artificial Neural Network)

이번 장에서는 딥 러닝의 출발점인 **인공 신경망(Artificial Neural Network)**을 다룬다. 앞에서 배운 선형 회귀·로지스틱 회귀·소프트맥스 회귀가 결국 인공 신경망의 한 형태였음을 확인하고, 퍼셉트론에서 출발하여 다층 퍼셉트론(MLP), 역전파, 그리고 깊은 신경망을 안정적으로 학습시키기 위한 여러 기법까지 순서대로 정리한다.

> **이 장의 흐름** : 머신 러닝 기본 개념 → 퍼셉트론 → XOR 문제로 보는 단층/다층의 차이 → 역전파(가중치가 어떻게 학습되는가) → 손글씨·MNIST 분류 실습 → 과적합을 막는 방법 → 기울기 소실/폭주를 막는 방법.
>
> 핵심 관통선은 앞 장들과 같다. 신경망의 학습은 **순전파로 예측 → 손실 함수로 오차 계산 → 역전파로 기울기 계산 → 옵티마이저로 가중치 갱신**을 반복하는 것이다. 달라지는 것은 모델의 구조(층의 깊이)일 뿐, 학습의 골격은 동일하다.

# 1. 머신 러닝 기본 개념 정리하기

본격적으로 인공 신경망을 배우기 전에, 머신 러닝의 공통 개념들을 정리한다. 딥 러닝 또한 머신 러닝에 속하므로 아래 내용은 모두 딥 러닝의 특징이기도 하다.

## 머신 러닝 모델의 평가

모델을 평가하기 위해 데이터를 보통 **훈련용(Training)**, **검증용(Validation)**, **테스트용(Testing)** 세 가지로 나눈다. (개념 학습 목적의 일부 실습에서는 검증용을 생략하고 훈련용·테스트용으로만 나누기도 한다.)

그렇다면 검증용 데이터는 왜 따로 두는가? 검증용 데이터는 모델의 **성능을 평가**하기 위한 것이 아니라 **성능을 조정**하기 위한 것이다. 더 정확히는 과적합 여부를 판단하거나 **하이퍼파라미터(hyperparameter)**를 튜닝하기 위한 용도다.

> - **하이퍼파라미터** : 값에 따라 모델 성능에 영향을 주며, **사람이 직접 정하는** 변수. 학습률, 은닉층의 수, 뉴런의 수, 드롭아웃 비율 등이 이에 해당한다.
> - **매개변수(parameter)** : 가중치·편향처럼 **기계가 학습 과정에서 스스로 찾는** 변수.

훈련을 마친 모델은 검증용 데이터로 하이퍼파라미터를 튜닝한다. 이 과정에서 모델은 점차 검증용 데이터에도 맞춰지므로, 튜닝이 끝난 뒤의 최종 성능 평가는 모델이 한 번도 보지 못한 **테스트 데이터**로 해야 한다. 비유하자면 훈련 데이터는 문제지, 검증 데이터는 모의고사, 테스트 데이터는 수능 시험이다. 데이터가 부족해 검증/테스트를 나누기 어렵다면 **k-폴드 교차 검증(k-fold cross validation)**을 쓰기도 한다.

## 분류(Classification)와 회귀(Regression)

머신 러닝의 많은 문제는 분류 또는 회귀에 속한다.

- **분류** : 정해진 선택지(클래스) 중 하나를 고르는 문제. 둘 중 하나면 **이진 분류(Binary Classification)**, 세 개 이상이면 **다중 클래스 분류(Multi-Class Classification)**다. (앞 장의 로지스틱 회귀가 이진 분류, 소프트맥스 회귀가 다중 클래스 분류였다.)
- **회귀** : 0/1이나 카테고리처럼 분리된 값이 아니라 **연속된 값**을 예측하는 문제. 공부 시간으로 시험 점수를 예측하거나, 시계열로 주가를 예측하는 문제 등이 이에 속한다.

## 지도 학습과 비지도 학습

머신 러닝은 크게 지도 학습, 비지도 학습, 강화 학습으로 나뉜다. (강화 학습은 이 책의 범위를 벗어난다.)

- **지도 학습(Supervised Learning)** : **레이블(Label)**이라는 정답과 함께 학습한다. 기계는 예측값 $\hat{y}$ 와 실제값 $y$ 의 차이인 오차를 줄이는 방향으로 학습한다.
- **비지도 학습(Unsupervised Learning)** : 레이블이 없는 학습. 군집(clustering)이나 차원 축소가 대표적이다.

## 샘플(Sample)과 특성(Feature)

많은 머신 러닝 문제는 1개 이상의 독립 변수 $x$ 로 종속 변수 $y$ 를 예측한다. 인공 신경망은 이를 행렬 연산으로 처리하므로 훈련 데이터를 행렬로 표현하는 경우가 많다. 독립 변수가 $n$ 개, 데이터가 $m$ 개인 행렬 $X$ 에서,

- **샘플(Sample)** : 하나의 데이터, 즉 행렬의 한 **행**. (데이터베이스의 레코드에 해당)
- **특성(Feature)** : 종속 변수 $y$ 를 예측하기 위한 각각의 독립 변수, 즉 한 **열**.

## 혼동 행렬(Confusion Matrix)

맞춘 문제 수를 전체 문제 수로 나눈 값을 **정확도(Accuracy)**라 한다. 하지만 정확도만으로는 맞춘/틀린 결과의 세부를 알 수 없다. 이를 위해 **혼동 행렬**을 사용한다. 각 열은 예측값, 각 행은 실제값을 나타낸다.

| 실제 \\ 예측 | 참(Positive) | 거짓(Negative) |
|:---:|:---:|:---:|
| 참(Positive) | TP | FN |
| 거짓(Negative) | FP | TN |

- **TP(True Positive)** : 양성이라 예측했고 실제로 양성(정답)
- **TN(True Negative)** : 음성이라 예측했고 실제로 음성(정답)
- **FP(False Positive)** : 양성이라 예측했으나 실제로 음성(오답)
- **FN(False Negative)** : 음성이라 예측했으나 실제로 양성(오답)

이 개념으로 정밀도와 재현율을 정의한다.

- **정밀도(Precision)** : 양성이라 **예측한** 것 중 실제 양성의 비율.

$$ Precision = \frac{TP}{TP + FP} $$

- **재현율(Recall)** : 실제 양성인 것 중 양성이라 **맞힌** 비율.

$$ Recall = \frac{TP}{TP + FN} $$

## 과적합(Overfitting)과 과소적합(Underfitting)

같은 문제지를 너무 많이 풀어서 문제 번호만 봐도 답을 맞히게 되었다고 하자. 그런데 다른 시험에서 점수가 안 좋다면 의미가 없다.

- **과적합(Overfitting)** : 훈련 데이터를 과하게 학습한 경우. 훈련 데이터에 대한 오차는 낮지만, 테스트 데이터에 대한 오차는 높다. 훈련 데이터에 포함된 노이즈까지 암기한 상태다.
- **과소적합(Underfitting)** : 훈련 자체가 부족한 상태. 과적합과 달리 훈련 데이터에 대해서도 정확도가 낮다.

> 에포크(epoch)는 전체 훈련 데이터에 대한 훈련 횟수다. 학습이 진행되면 훈련 오차는 계속 낮아지지만, 어느 시점부터 테스트 오차는 오히려 증가하기 시작한다(과적합 시작). 따라서 테스트 오차가 증가하기 전에 훈련을 멈추는 것이 바람직하며, 이를 **조기 종료(Early Stopping)**라 한다. (과적합을 막는 방법들은 이 장의 7절에서 자세히 다룬다.)

## 비선형 활성화 함수 (Activation Function)

**활성화 함수**는 입력을 받아 수학적 변환을 수행하고 출력을 생성하는 함수다. 앞서 배운 시그모이드·소프트맥스 함수도 대표적인 활성화 함수다.

### 왜 비선형 함수여야 하는가

활성화 함수는 반드시 **비선형 함수**여야 한다. 선형 함수란 출력이 입력의 상수배만큼 변하는 함수($f(x) = Wx + b$, 그래프가 직선)를 말한다.

만약 활성화 함수로 선형 함수 $f(x) = Wx$ 를 쓰고 은닉층을 여러 번 쌓는다고 하자. 출력층을 포함해 3번 쌓으면 $y(x) = f(f(f(x))) = W \times W \times W \times x$ 가 되는데, $W^3$ 를 새로운 상수 $k$ 로 정의하면 $y(x) = kx$ 와 같다. 즉 **선형 함수로는 은닉층을 아무리 쌓아도 1회 쌓은 것과 차이가 없다.** 신경망의 표현력을 높이려면 비선형 활성화 함수가 필수다.

> 선형 함수를 쓴 층이 무의미하다는 뜻은 아니다. 학습 가능한 가중치가 새로 생긴다는 점에서 의미가 있으며, 이런 층을 활성화 함수를 쓰는 은닉층과 구분해 **선형층(linear layer)** 또는 **투사층(projection layer)**이라 부른다.

### 시그모이드 함수와 기울기 소실

시그모이드 함수의 그래프를 직접 그려 본다. 먼저 도구를 임포트한다.

In [ ]:
import numpy as np  # 넘파이 사용
import matplotlib.pyplot as plt  # 맷플롯립 사용

In [ ]:
# 시그모이드 함수 그래프를 그리는 코드
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.arange(-5.0, 5.0, 0.1)
y = sigmoid(x)

plt.plot(x, y)
plt.plot([0, 0], [1.0, 0.0], ':')  # 가운데 점선 추가
plt.title('Sigmoid Function')
plt.show()

시그모이드 함수의 출력값이 0 또는 1에 가까워지면 그래프의 기울기가 완만해진다. 이 구간에서 기울기를 계산하면 0에 가까운 아주 작은 값이 나온다.

문제는 역전파 과정에서 발생한다. 역전파는 출력층에서 입력층 방향으로 기울기를 곱해 가며 전달하는데, 0에 가까운 기울기가 계속 곱해지면 앞단(입력층 쪽)에는 기울기가 거의 전달되지 않는다. 이를 **기울기 소실(Vanishing Gradient)** 문제라 한다. 시그모이드를 쓰는 은닉층이 깊어질수록 앞단의 매개변수 $W$ 가 업데이트되지 않아 학습이 되지 않는다.

> **결론 : 은닉층의 활성화 함수로 시그모이드 함수는 지양한다.** 또한 시그모이드는 출력이 항상 양수라 원점 중심(zero-centered)이 아니라는 문제도 있어, 층을 지날수록 편향 이동(bias shift)이 누적될 수 있다.

### 하이퍼볼릭탄젠트 함수 (tanh)

하이퍼볼릭탄젠트 함수는 입력을 -1과 1 사이로 변환한다.

In [ ]:
x = np.arange(-5.0, 5.0, 0.1)
y = np.tanh(x)

plt.plot(x, y)
plt.plot([0, 0], [1.0, -1.0], ':')
plt.axhline(y=0, color='orange', linestyle='--')
plt.title('Tanh Function')
plt.show()

tanh도 -1, 1에 가까운 값에서 기울기가 완만해져 시그모이드와 같은 기울기 소실 문제가 있다. 하지만 시그모이드와 달리 **원점을 중심**으로 하여 반환값의 변화폭이 더 크므로, 기울기 소실 증상이 상대적으로 적다. 그래서 은닉층에서 시그모이드보다는 많이 쓰인다.

### 렐루 함수 (ReLU)

인공 신경망에서 가장 인기 있는 활성화 함수다. 수식은 $f(x) = max(0, x)$ 로 아주 간단하다.

In [ ]:
def relu(x):
    return np.maximum(0, x)

x = np.arange(-5.0, 5.0, 0.1)
y = relu(x)

plt.plot(x, y)
plt.plot([0, 0], [5.0, 0.0], ':')
plt.title('Relu Function')
plt.show()

ReLU는 음수를 입력하면 0을, 양수를 입력하면 입력값을 그대로 반환한다. 특정 양수값에 수렴하지 않으므로 깊은 신경망에서 시그모이드보다 훨씬 잘 작동하고, 단순 임계값 연산이라 속도도 빠르다.

다만 입력값이 음수면 기울기도 0이 되고, 이 뉴런은 다시 회생하기 어렵다. 이를 **죽은 렐루(dying ReLU)** 문제라 한다.

### 리키 렐루 (Leaky ReLU)

죽은 렐루를 보완한 변형이다. 입력값이 음수일 때 0이 아니라 아주 작은 값을 반환한다. 수식은 $f(x) = max(ax, x)$ 이며, $a$ 는 '새는(leaky) 정도'를 결정하는 하이퍼파라미터로 보통 0.01을 쓴다. (아래에서는 새는 모습을 잘 보이려고 0.1로 잡았다.)

In [ ]:
a = 0.1

def leaky_relu(x):
    return np.maximum(a * x, x)

x = np.arange(-5.0, 5.0, 0.1)
y = leaky_relu(x)

plt.plot(x, y)
plt.plot([0, 0], [5.0, 0.0], ':')
plt.title('Leaky ReLU Function')
plt.show()

입력값이 음수라도 기울기가 0이 되지 않으므로 ReLU는 죽지 않는다.

### 소프트맥스 함수 (Softmax)

은닉층에서는 ReLU 계열을 쓰는 것이 일반적이지만, 출력층에서는 여전히 시그모이드·소프트맥스가 쓰인다. 소프트맥스는 시그모이드처럼 출력층에서 사용되며, 시그모이드가 이진 분류에 쓰인다면 소프트맥스는 세 개 이상의 상호 배타적 선택지 중 하나를 고르는 다중 클래스 분류에 쓰인다.

In [ ]:
x = np.arange(-5.0, 5.0, 0.1)
y = np.exp(x) / np.sum(np.exp(x))

plt.plot(x, y)
plt.title('Softmax Function')
plt.show()

### 출력층의 활성화 함수와 오차 함수의 관계

은닉층은 ReLU 또는 Leaky ReLU 계열을 쓴다고 정리했다. 그렇다면 출력층은? 문제 유형에 따른 출력층 활성화 함수와 비용 함수의 관계는 다음과 같다.

| 문제 | 활성화 함수 | 비용 함수 |
|:---|:---|:---|
| 이진 분류 | 시그모이드 | `nn.BCELoss()` |
| 다중 클래스 분류 | 소프트맥스 | `nn.CrossEntropyLoss()` |
| 회귀 | 없음 | MSE |

> **여기서 잠깐!** `nn.CrossEntropyLoss()`(그리고 `F.cross_entropy()`)는 **소프트맥스 함수를 이미 포함**하고 있다. 따라서 다중 클래스 분류 모델의 출력층(가설)에서 소프트맥스를 또 적용하지 않도록 주의한다. (스탠포드 cs231n에서는 ReLU를 먼저 시도하고, 그다음 Leaky ReLU·ELU 같은 변형을 시도하며, 시그모이드는 쓰지 말라고 권장한다.)

# 2. 퍼셉트론 (Perceptron)

이제 초기의 인공 신경망인 **퍼셉트론**을 이해한다. 딥 러닝을 이해하려면 먼저 인공 신경망에 대한 이해가 필요하다.

## 퍼셉트론이란

퍼셉트론은 프랑크 로젠블라트가 1957년에 제안한 초기 인공 신경망으로, 다수의 입력으로부터 하나의 결과를 내보내는 알고리즘이다. 신경 세포 뉴런이 가지돌기에서 신호를 받아 일정치 이상이면 축삭돌기로 신호를 전달하는 동작과 유사하다.

$x$ 는 입력값, $W$ 는 **가중치(Weight)**, $y$ 는 출력값이다. 실제 뉴런에서 축삭돌기의 역할을 퍼셉트론에서는 가중치가 대신한다. 각 입력값 $x$ 는 가중치 $W$ 와 곱해져 인공 뉴런에 전달된다. 가중치 값이 클수록 해당 입력이 중요하다는 의미다.

각 입력값과 가중치의 곱의 전체 합이 **임계치(threshold) $\theta$**를 넘으면 뉴런은 1을, 넘지 못하면 0을 출력한다. 이러한 함수를 **계단 함수(Step function)**라 한다.

$$ \text{if } \sum_{i}^{n} W_i x_i \geq \theta \rightarrow y = 1 $$
$$ \text{if } \sum_{i}^{n} W_i x_i < \theta \rightarrow y = 0 $$

임계치를 좌변으로 넘기면 **편향(bias) $b$**로 표현할 수 있다. 편향 또한 퍼셉트론의 입력으로 사용되며(입력값 1에 곱해지는 변수), 딥 러닝이 찾아야 할 변수 중 하나다.

$$ \text{if } \sum_{i}^{n} W_i x_i + b \geq 0 \rightarrow y = 1 $$
$$ \text{if } \sum_{i}^{n} W_i x_i + b < 0 \rightarrow y = 0 $$

> 퍼셉트론의 활성화 함수는 계단 함수이지만, 이를 시그모이드 함수로 바꾸면 곧 **로지스틱 회귀**와 동일해진다. 즉 로지스틱 회귀 모델은 인공 신경망에서 하나의 인공 뉴런으로 볼 수 있으며, 둘의 차이는 오직 활성화 함수의 차이다. (퍼셉트론을 배우기 전에 로지스틱 회귀를 먼저 배운 이유가 여기에 있다.)

## 단층 퍼셉트론 (Single-Layer Perceptron)

퍼셉트론은 단층과 다층으로 나뉜다. **단층 퍼셉트론**은 값을 보내는 단계(입력층)와 값을 받아 출력하는 단계(출력층) **두 개의 층**으로만 이루어진다.

단층 퍼셉트론으로 AND, NAND, OR 게이트를 구현할 수 있다. 게이트 연산은 두 입력값과 하나의 출력값을 쓴다. 먼저 AND 게이트는 두 입력이 모두 1일 때만 1을 출력한다.

| $x_1$ | $x_2$ | $y$ |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

AND 게이트를 만족하는 가중치·편향 $[w_1, w_2, b]$ 조합은 $[0.5, 0.5, -0.7]$, $[0.5, 0.5, -0.8]$, $[1.0, 1.0, -1.0]$ 등 다양하다. 파이썬으로 구현해 본다.

In [ ]:
def AND_gate(x1, x2):
    w1 = 0.5
    w2 = 0.5
    b = -0.7
    result = x1 * w1 + x2 * w2 + b
    if result <= 0:
        return 0
    else:
        return 1

AND_gate(0, 0), AND_gate(0, 1), AND_gate(1, 0), AND_gate(1, 1)

두 입력이 모두 1인 경우에만 1을 출력한다. 다음으로 NAND 게이트는 두 입력이 모두 1일 때만 0, 나머지는 1을 출력한다. AND 게이트의 $[0.5, 0.5, -0.7]$ 에 부호를 뒤집은 $[-0.5, -0.5, +0.7]$ 을 넣으면 NAND를 만족한다. (구조는 같고 가중치·편향만 바꿨다.)

In [ ]:
def NAND_gate(x1, x2):
    w1 = -0.5
    w2 = -0.5
    b = 0.7
    result = x1 * w1 + x2 * w2 + b
    if result <= 0:
        return 0
    else:
        return 1

NAND_gate(0, 0), NAND_gate(0, 1), NAND_gate(1, 0), NAND_gate(1, 1)

OR 게이트는 두 입력이 모두 0일 때만 0, 나머지는 1을 출력한다. 가중치·편향으로 $[0.6, 0.6, -0.5]$ 를 선택하면 OR을 만족한다.

In [ ]:
def OR_gate(x1, x2):
    w1 = 0.6
    w2 = 0.6
    b = -0.5
    result = x1 * w1 + x2 * w2 + b
    if result <= 0:
        return 0
    else:
        return 1

OR_gate(0, 0), OR_gate(0, 1), OR_gate(1, 0), OR_gate(1, 1)

## 단층 퍼셉트론의 한계 : XOR 게이트

단층 퍼셉트론은 AND·NAND·OR은 구현할 수 있지만 **XOR 게이트는 구현할 수 없다.** XOR은 두 입력이 서로 다를 때만 1, 같으면 0을 출력한다.

| $x_1$ | $x_2$ | $y$ |
|:---:|:---:|:---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

위의 파이썬 코드에 아무리 많은 가중치와 편향을 넣어도 XOR을 구현하는 것은 불가능하다. 그 이유는 **단층 퍼셉트론은 직선 하나로 두 영역을 나눌 수 있는(선형 분리 가능한) 문제만** 풀 수 있기 때문이다.

AND·NAND·OR은 출력 0인 점과 1인 점을 직선 하나로 나눌 수 있다. 하지만 XOR은 (0,0), (1,1)이 한 클래스, (0,1), (1,0)이 다른 클래스라서 어떤 직선을 그어도 두 클래스를 나눌 수 없다. XOR은 직선이 아닌 **곡선(비선형 영역)**으로만 분리할 수 있다.

## 다층 퍼셉트론 (MultiLayer Perceptron, MLP)

XOR 게이트는 AND, NAND, OR 게이트를 **조합**하면 만들 수 있다. 퍼셉트론 관점에서는 **층을 더 쌓으면** 된다. 입력층과 출력층 사이에 추가되는 층을 **은닉층(hidden layer)**이라 하며, 은닉층이 존재하는 퍼셉트론이 다층 퍼셉트론(MLP)이다.

> XOR은 은닉층 1개로 해결되지만, 다층 퍼셉트론은 본래 은닉층이 1개 이상인 퍼셉트론을 말한다. 더 복잡한 문제를 풀려면 은닉층을 수십 개까지 늘릴 수 있다. **은닉층이 2개 이상인 신경망을 심층 신경망(Deep Neural Network, DNN)**이라 하며, 이를 학습시키는 것을 **딥 러닝(Deep Learning)**이라 한다.

지금까지 게이트 예제에서는 정답을 보고 가중치를 **수동으로** 찾았다. 하지만 실제로는 기계가 가중치를 **스스로** 찾아내도록 자동화해야 하는데, 이것이 학습(training) 단계다. 앞 장의 선형 회귀·로지스틱 회귀처럼 **손실 함수(Loss function)**와 **옵티마이저(Optimizer)**를 사용한다. 다음 절에서 파이토치로 단층·다층 퍼셉트론을 직접 구현해 XOR을 풀어 본다.

# 3. XOR 문제 구현하기 - 단층 퍼셉트론과 다층 퍼셉트론

파이토치로 단층 퍼셉트론과 다층 퍼셉트론을 각각 구현해 XOR 문제를 풀어 본다. 단층으로는 풀 수 없고 다층으로는 풀 수 있음을 직접 확인한다.

## 단층 퍼셉트론으로 XOR 풀기 (실패)

필요한 도구를 임포트하고 GPU 연산이 가능하면 GPU를 쓰도록 설정한다.

In [ ]:
import torch
import torch.nn as nn

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)

XOR 문제에 해당하는 입력 X와 출력 Y를 정의한다.

In [ ]:
X = torch.FloatTensor([[0, 0], [0, 1], [1, 0], [1, 1]]).to(device)
Y = torch.FloatTensor([[0], [1], [1], [0]]).to(device)

1개의 뉴런을 가지는 단층 퍼셉트론을 구현한다. 활성화 함수로는 계단 함수 대신 시그모이드 함수를 쓴다. 0 또는 1을 예측하는 이진 분류이므로 비용 함수로 크로스 엔트로피 함수인 `nn.BCELoss()` 를 사용한다.

In [ ]:
linear = nn.Linear(2, 1, bias=True)
sigmoid = nn.Sigmoid()
model = nn.Sequential(linear, sigmoid).to(device)

# 비용 함수와 옵티마이저 정의
criterion = torch.nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1)

In [ ]:
for step in range(10001):
    optimizer.zero_grad()
    hypothesis = model(X)

    # 비용 함수
    cost = criterion(hypothesis, Y)
    cost.backward()
    optimizer.step()

    if step % 100 == 0:  # 100번째 에포크마다 비용 출력
        print(step, cost.item())

200번 에포크에서 비용이 약 0.6931로 출력된 이후 10,000번까지 더 이상 줄어들지 않는다. 단층 퍼셉트론이 XOR 문제를 풀 수 없기 때문이다. 학습된 모델의 예측값을 확인해 본다.

> `with torch.no_grad():` 블록 안에서는 기울기 계산을 비활성화해 연산 속도를 높이고 메모리 사용을 줄인다. 평가 단계에서는 기울기 계산이 필요 없기 때문이다. 예측값 `hypothesis` 가 0.5를 초과하면 1, 아니면 0으로 간주해 이진 분류를 수행한다.

In [ ]:
with torch.no_grad():
    hypothesis = model(X)
    predicted = (hypothesis > 0.5).float()
    accuracy = (predicted == Y).float().mean()
    print('모델의 출력값(Hypothesis): ', hypothesis.detach().cpu().numpy())
    print('모델의 예측값(Predicted): ', predicted.detach().cpu().numpy())
    print('실제값(Y): ', Y.cpu().numpy())
    print('정확도(Accuracy): ', accuracy.item())

실제값은 0, 1, 1, 0이지만 예측값은 모두 0.5 부근이라 0, 0, 0, 0으로 나오며 정확도는 0.5에 그친다. 단층 퍼셉트론은 XOR을 풀지 못한다.

## 다층 퍼셉트론으로 XOR 풀기 (성공)

이번에는 은닉층을 추가한 다층 퍼셉트론으로 XOR을 푼다. 입력과 출력 정의는 동일하다. 아래는 입력층, 은닉층1·2·3, 출력층을 가지는 **은닉층이 3개인** 인공 신경망이다. 각 은닉층의 활성화 함수는 시그모이드다.

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 10, bias=True),   # input_layer = 2, hidden_layer1 = 10
    nn.Sigmoid(),
    nn.Linear(10, 10, bias=True),  # hidden_layer1 = 10, hidden_layer2 = 10
    nn.Sigmoid(),
    nn.Linear(10, 10, bias=True),  # hidden_layer2 = 10, hidden_layer3 = 10
    nn.Sigmoid(),
    nn.Linear(10, 1, bias=True),   # hidden_layer3 = 10, output_layer = 1
    nn.Sigmoid()
).to(device)

비용 함수와 옵티마이저를 선언한다. 이진 분류이므로 동일하게 `nn.BCELoss()` 를 쓴다.

In [ ]:
criterion = torch.nn.BCELoss().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1)  # learning rate를 0.1에서 1로 수정

총 10,001번의 에포크를 수행한다. 각 에포크마다 역전파가 수행된다고 보면 된다. 비용이 최소화되는 방향으로 가중치와 편향이 업데이트된다.

In [ ]:
for epoch in range(10001):
    optimizer.zero_grad()
    # forward 연산
    hypothesis = model(X)

    # 비용 함수
    cost = criterion(hypothesis, Y)
    cost.backward()
    optimizer.step()

    # 100의 배수에 해당되는 에포크마다 비용을 출력
    if epoch % 100 == 0:
        print(epoch, cost.item())

이번에는 비용이 0에 가깝게 줄어든다. 모델이 XOR을 풀 수 있는지 예측값으로 확인한다.

In [ ]:
with torch.no_grad():
    hypothesis = model(X)
    predicted = (hypothesis > 0.5).float()
    accuracy = (predicted == Y).float().mean()
    print('모델의 출력값(Hypothesis): ', hypothesis.detach().cpu().numpy())
    print('모델의 예측값(Predicted): ', predicted.detach().cpu().numpy())
    print('실제값(Y): ', Y.cpu().numpy())
    print('정확도(Accuracy): ', accuracy.item())

실제값 0, 1, 1, 0에 대해 예측값도 0, 1, 1, 0으로 나오며 정확도는 1.0이다. 다층 퍼셉트론은 XOR 문제를 해결한다.

# 4. 역전파 (BackPropagation)

지금까지는 파이토치가 `cost.backward()` 한 줄로 기울기를 알아서 계산해 주었다. 이번 절에서는 그 내부에서 무슨 일이 일어나는지, 즉 인공 신경망이 **순전파로 오차를 구한 뒤 역전파에서 경사 하강법으로 가중치를 어떻게 업데이트하는지** 직접 계산으로 이해한다.

## 인공 신경망의 구조

예제로 사용할 신경망은 입력층, 은닉층, 출력층 3개 층을 가진다. 입력 2개, 은닉층 뉴런 2개, 출력층 뉴런 2개를 사용하며, 은닉층·출력층의 모든 뉴런은 활성화 함수로 시그모이드를 쓴다.

- $z$ : 이전 층의 모든 입력이 가중치와 곱해져 더해진 **가중합**. 아직 시그모이드를 거치지 않은, 활성화 함수의 입력.
- $h$ 또는 $o$ : $z$ 가 시그모이드를 지난 후의 값, 즉 각 뉴런의 **출력값**.

목표는 신경망의 모든 가중치 $W_1 \sim W_8$ 을 역전파로 업데이트하는 것이다. (이 예제에서 편향 $b$ 는 고려하지 않는다.)

초기값은 다음과 같이 주어진다. 입력 $x_1 = 0.1$, $x_2 = 0.2$, 실제값 $target_{o1} = 0.4$, $target_{o2} = 0.6$. 가중치는 $W_1 = 0.3,\ W_2 = 0.25,\ W_3 = 0.4,\ W_4 = 0.35$ (입력층→은닉층), $W_5 = 0.45,\ W_6 = 0.4,\ W_7 = 0.7,\ W_8 = 0.6$ (은닉층→출력층)이다.

## 순전파 (Forward Propagation)

입력은 입력층에서 은닉층 방향으로 가중치와 곱해져 가중합으로 계산되고, 이것이 은닉층 뉴런의 시그모이드 입력이 된다. (계산 결과는 소수점 아래 여덟째 자리까지 반올림한다.)

$$ z_1 = W_1 x_1 + W_2 x_2 = 0.3 \times 0.1 + 0.25 \times 0.2 = 0.08 $$
$$ z_2 = W_3 x_1 + W_4 x_2 = 0.4 \times 0.1 + 0.35 \times 0.2 = 0.11 $$

$z_1, z_2$ 가 각 은닉층 뉴런에서 시그모이드를 지난 결과가 은닉층의 출력 $h_1, h_2$ 다.

$$ h_1 = sigmoid(z_1) = 0.51998934 $$
$$ h_2 = sigmoid(z_2) = 0.52747230 $$

$h_1, h_2$ 는 다시 출력층 뉴런으로 향해 가중치와 곱해지고 가중합되어 출력층 뉴런의 시그모이드 입력 $z_3, z_4$ 가 된다.

$$ z_3 = W_5 h_1 + W_6 h_2 = 0.45 \times h_1 + 0.4 \times h_2 = 0.44498412 $$
$$ z_4 = W_7 h_1 + W_8 h_2 = 0.7 \times h_1 + 0.6 \times h_2 = 0.68047592 $$

$z_3, z_4$ 가 시그모이드를 지난 값이 이 신경망의 최종 출력(예측값) $o_1, o_2$ 다.

$$ o_1 = sigmoid(z_3) = 0.60944600 $$
$$ o_2 = sigmoid(z_4) = 0.66384491 $$

이제 예측값과 실제값의 오차를 계산한다. 손실 함수로 평균 제곱 오차(MSE)를 쓰며, 각 오차를 더해 전체 오차 $E_{total}$ 을 구한다.

$$ E_{o1} = \frac{1}{2}(target_{o1} - output_{o1})^2 = \frac{1}{2}(0.4 - 0.60944600)^2 = 0.02193381 $$
$$ E_{o2} = \frac{1}{2}(target_{o2} - output_{o2})^2 = \frac{1}{2}(0.6 - 0.66384491)^2 = 0.00203809 $$
$$ E_{total} = E_{o1} + E_{o2} = 0.02397190 $$

## 역전파 1단계 (출력층 → 은닉층 가중치)

순전파가 입력층→출력층이라면 역전파는 반대로 출력층→입력층 방향으로 계산하며 가중치를 업데이트한다. 출력층과 바로 이전 은닉층 사이의 가중치($W_5, W_6, W_7, W_8$)를 업데이트하는 단계를 역전파 1단계라 한다.

원리는 동일하므로 $W_5$ 부터 본다. 경사 하강법으로 $W_5$ 를 업데이트하려면 $\frac{\partial E_{total}}{\partial W_5}$ 를 구해야 한다. 미분의 **연쇄 법칙(Chain rule)**에 따라 다음과 같이 분해한다.

$$ \frac{\partial E_{total}}{\partial W_5} = \frac{\partial E_{total}}{\partial o_1} \times \frac{\partial o_1}{\partial z_3} \times \frac{\partial z_3}{\partial W_5} $$

우변의 세 항을 순서대로 계산한다.

**첫째 항** : $E_{total} = \frac{1}{2}(target_{o1} - output_{o1})^2 + \frac{1}{2}(target_{o2} - output_{o2})^2$ 이므로,

$$ \frac{\partial E_{total}}{\partial o_1} = -(target_{o1} - output_{o1}) = -(0.4 - 0.60944600) = 0.20944600 $$

**둘째 항** : $o_1$ 은 시그모이드 출력이고, 시그모이드 함수의 미분은 $f(x)(1 - f(x))$ 다. (앞으로 자주 쓰이므로 기억해 둔다.)

$$ \frac{\partial o_1}{\partial z_3} = o_1 \times (1 - o_1) = 0.60944600 \times (1 - 0.60944600) = 0.23802157 $$

**셋째 항** : $z_3 = W_5 h_1 + W_6 h_2$ 이므로 $W_5$ 로 미분하면 $h_1$ 만 남는다.

$$ \frac{\partial z_3}{\partial W_5} = h_1 = 0.51998934 $$

세 항을 모두 곱한다.

$$ \frac{\partial E_{total}}{\partial W_5} = 0.20944600 \times 0.23802157 \times 0.51998934 = 0.02592286 $$

이제 경사 하강법으로 가중치를 업데이트한다. 학습률 $\alpha$ 는 0.5로 가정한다.

$$ W_5^{+} = W_5 - \alpha \frac{\partial E_{total}}{\partial W_5} = 0.45 - 0.5 \times 0.02592286 = 0.43703857 $$

같은 원리로 나머지 가중치도 계산한다.

$$ W_6^{+} = 0.38685205,\quad W_7^{+} = 0.69629578,\quad W_8^{+} = 0.59624247 $$

## 역전파 2단계 (은닉층 → 입력층 가중치)

1단계를 마쳤으면 입력층 방향으로 이동해 $W_1, W_2, W_3, W_4$ 를 업데이트한다. (현재 신경망은 은닉층이 1개라 이번이 마지막 단계다. 은닉층이 더 많으면 입력층 방향으로 한 단계씩 계속 계산한다.)

$W_1$ 부터 본다. 연쇄 법칙으로 분해한다.

$$ \frac{\partial E_{total}}{\partial W_1} = \frac{\partial E_{total}}{\partial h_1} \times \frac{\partial h_1}{\partial z_1} \times \frac{\partial z_1}{\partial W_1} $$

첫째 항 $\frac{\partial E_{total}}{\partial h_1}$ 은 $h_1$ 이 $o_1, o_2$ 양쪽에 영향을 주므로 두 경로의 합으로 다시 쓸 수 있다.

$$ \frac{\partial E_{total}}{\partial h_1} = \frac{\partial E_{o1}}{\partial h_1} + \frac{\partial E_{o2}}{\partial h_1} $$

각 항을 연쇄 법칙으로 풀면 다음과 같다.

$$ \frac{\partial E_{o1}}{\partial h_1} = \frac{\partial E_{o1}}{\partial o_1} \times \frac{\partial o_1}{\partial z_3} \times \frac{\partial z_3}{\partial h_1} = -(target_{o1} - output_{o1}) \times o_1(1 - o_1) \times W_5 = 0.02243370 $$
$$ \frac{\partial E_{o2}}{\partial h_1} = 0.00997311 $$
$$ \frac{\partial E_{total}}{\partial h_1} = 0.02243370 + 0.00997311 = 0.03240681 $$

나머지 두 항은 다음과 같다.

$$ \frac{\partial h_1}{\partial z_1} = h_1(1 - h_1) = 0.51998934 \times (1 - 0.51998934) = 0.24960043 $$
$$ \frac{\partial z_1}{\partial W_1} = x_1 = 0.1 $$

세 항을 곱하고 경사 하강법으로 업데이트한다.

$$ \frac{\partial E_{total}}{\partial W_1} = 0.03240681 \times 0.24960043 \times 0.1 = 0.00080888 $$
$$ W_1^{+} = W_1 - \alpha \frac{\partial E_{total}}{\partial W_1} = 0.1 - 0.5 \times 0.00080888 = 0.29959556 $$

같은 원리로 나머지도 구한다.

$$ W_2^{+} = 0.24919112,\quad W_3^{+} = 0.39964496,\quad W_4^{+} = 0.34928991 $$

## 결과 확인

업데이트된 가중치로 다시 순전파를 진행해 오차가 줄었는지 확인한다.

$$ z_1 = 0.07979778,\quad z_2 = 0.10982248 $$
$$ h_1 = 0.51993887,\quad h_2 = 0.52742806 $$
$$ z_3 = 0.43126996,\quad z_4 = 0.67650625 $$
$$ o_1 = 0.60617688,\quad o_2 = 0.66295848 $$
$$ E_{o1} = 0.02125445,\quad E_{o2} = 0.00198189,\quad E_{total} = 0.02323634 $$

기존 전체 오차가 0.02397190이었으므로, **1번의 역전파로 오차가 0.02323634로 감소**했다. 인공 신경망의 학습이란 이처럼 순전파와 역전파를 반복하며 오차를 최소화하는 가중치를 찾는 과정이다.

# 5. 다층 퍼셉트론으로 손글씨 분류하기

다층 퍼셉트론을 구현해 숫자 필기 데이터를 분류한다. (다음 절의 MNIST와는 다른 데이터다.)

## 숫자 필기 데이터 소개

숫자 필기 데이터는 사이킷런이 제공하는 분류용 예제 데이터로, `load_digits()` 로 로드한다. 0부터 9까지 손으로 쓴 이미지이며, 각 이미지는 0~15 명암의 **8 × 8 = 64 픽셀** 흑백 이미지다. 총 1,797개의 샘플이 있다.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt  # 시각화를 위한 맷플롯립
from sklearn.datasets import load_digits
digits = load_digits()  # 1,797개의 이미지 데이터 로드

첫 번째 샘플을 출력해 본다. `.images[인덱스]` 는 해당 이미지를 행렬로 보여 준다.

In [ ]:
print(digits.images[0])

첫 번째 샘플이 8 × 8 행렬로 출력된다. 0을 흰색, 0보다 큰 값을 검은 점이라고 상상하면 숫자 0의 실루엣처럼 보인다. 실제 레이블을 확인한다.

In [ ]:
print(digits.target[0])

전체 샘플 개수와 상위 5개 샘플을 시각화한다.

In [ ]:
print('전체 샘플의 수 : {}'.format(len(digits.images)))

In [ ]:
images_and_labels = list(zip(digits.images, digits.target))
for index, (image, label) in enumerate(images_and_labels[:5]):  # 5개의 샘플만 출력
    plt.subplot(2, 5, index + 1)
    plt.axis('off')
    plt.imshow(image, cmap=plt.cm.gray_r, interpolation='nearest')
    plt.title('sample: %i' % label)

순서대로 숫자 0, 1, 2, 3, 4의 손글씨처럼 보인다. 훈련에는 `digits.images`(8×8 행렬)보다 이를 64차원 벡터로 펼친 `digits.data` 를 쓰는 것이 편하다. 이를 특성 행렬 X, 레이블을 Y에 저장한다.

In [ ]:
X = digits.data  # 이미지. 즉, 특성 행렬
Y = digits.target  # 각 이미지에 대한 레이블

## 다층 퍼셉트론 분류기 만들기

`nn.Sequential` 로 레이어를 순차적으로 쌓는다. 입력 특성이 64개이므로 첫 레이어는 64를 입력받아 32로, 다음은 32→16, 마지막 출력층은 16→10(클래스 개수)으로 변환한다. 각 은닉층 뒤에는 비선형성을 위해 ReLU를 둔다.

In [ ]:
import torch
import torch.nn as nn
from torch import optim

In [ ]:
# 모델 정의: 순차적인 레이어 구조
model = nn.Sequential(
    nn.Linear(64, 32),  # 입력층: 64, 첫 번째 은닉층: 32
    nn.ReLU(),          # 활성화 함수: ReLU
    nn.Linear(32, 16),  # 첫 번째 은닉층: 32, 두 번째 은닉층: 16
    nn.ReLU(),          # 활성화 함수: ReLU
    nn.Linear(16, 10)   # 두 번째 은닉층: 16, 출력층: 10 (클래스의 개수)
)

# 입력 데이터 X와 레이블 Y를 텐서로 변환
X = torch.tensor(X, dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.int64)

비용 함수로 `nn.CrossEntropyLoss()` 를 쓴다(소프트맥스를 포함하므로 출력층에 소프트맥스를 따로 두지 않는다). 옵티마이저는 Adam이다. 100번의 에포크 동안 학습하며 손실을 기록한다.

In [ ]:
loss_fn = nn.CrossEntropyLoss()  # 이 비용 함수는 소프트맥스 함수를 포함하고 있음.
optimizer = optim.Adam(model.parameters())

losses = []

# 총 100번의 에포크 동안 모델 학습
for epoch in range(100):
    optimizer.zero_grad()       # 옵티마이저의 기울기 초기화
    y_pred = model(X)           # 순전파 연산으로 예측값 계산
    loss = loss_fn(y_pred, Y)   # 손실 함수로 비용 계산
    loss.backward()             # 역전파 연산으로 기울기 계산
    optimizer.step()            # 옵티마이저를 통해 파라미터 업데이트

    if epoch % 10 == 0:
        print('Epoch {:4d}/{} Cost: {:.6f}'.format(epoch, 100, loss.item()))

    losses.append(loss.item())  # 손실 값을 리스트에 추가하여 추적

에포크가 진행될수록 손실이 줄어든다. 손실 값의 변화를 그래프로 확인한다.

In [ ]:
plt.plot(losses)

# 6. 다층 퍼셉트론으로 MNIST 분류하기

앞 장에서 소프트맥스 회귀('단층 퍼셉트론')로 MNIST를 분류했다. 이번에는 은닉층을 추가한 **다층 퍼셉트론**으로 MNIST를 분류한다.

## 데이터 로드하기

`fetch_openml` 로 MNIST 데이터셋을 불러온다. `cache=True` 는 다운로드한 데이터를 로컬에 저장해 다음 호출을 빠르게 한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', version=1, cache=True, as_frame=False)

레이블은 현재 문자열 타입이므로 정수형으로 변환하고, 0~255 범위의 픽셀값을 0~1로 정규화한다(학습 효율을 높이기 위함). 특성 행렬을 X, 레이블을 y에 저장한다.

In [ ]:
mnist.target = mnist.target.astype(np.int8)
X = mnist.data / 255  # 0-255값을 [0,1] 구간으로 정규화
y = mnist.target

첫 번째 샘플을 시각화하고 레이블을 출력한다.

In [ ]:
plt.imshow(X[0].reshape(28, 28), cmap='gray')
print("이 이미지 데이터의 레이블은 {:.0f}이다".format(y[0]))

## 훈련 데이터와 테스트 데이터의 분리

`train_test_split` 으로 데이터를 분할한 뒤 파이토치 텐서로 변환하고, `TensorDataset` 으로 데이터셋을 구성한다. 이를 `DataLoader` 에 적용하면 미니 배치 단위로 편리하게 데이터를 로드할 수 있다.

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=1/7, random_state=0)

# 텐서로 변환
X_train = torch.Tensor(X_train)
X_test = torch.Tensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

# TensorDataset 객체 생성
ds_train = TensorDataset(X_train, y_train)
ds_test = TensorDataset(X_test, y_test)

# DataLoader 객체 생성
loader_train = DataLoader(ds_train, batch_size=64, shuffle=True)
loader_test = DataLoader(ds_test, batch_size=64, shuffle=False)

## 다층 퍼셉트론 설계 및 학습

784(28×28)개의 입력 뉴런, 100개의 은닉 뉴런, 10개의 출력 뉴런(0~9)을 가지는 다층 퍼셉트론을 구성한다. ReLU로 비선형성을 더하고, 최종 출력은 10개 클래스에 대한 점수를 제공한다.

In [ ]:
from torch import nn
from torch import optim

model = nn.Sequential()
model.add_module('fc1', nn.Linear(28 * 28 * 1, 100))
model.add_module('relu1', nn.ReLU())
model.add_module('fc2', nn.Linear(100, 100))
model.add_module('relu2', nn.ReLU())
model.add_module('fc3', nn.Linear(100, 10))
print(model)

크로스 엔트로피 손실 함수로 예측과 실제 레이블의 차이를 계산하고, Adam으로 가중치를 업데이트한다. 데이터 로더에서 미니 배치를 하나씩 꺼내 순전파→손실 계산→역전파→파라미터 갱신을 수행한다.

In [ ]:
# 오차함수 선택
loss_fn = nn.CrossEntropyLoss()
# 가중치를 학습하기 위한 최적화 기법 선택
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 총 3번의 에포크 동안 모델 학습
epochs = 3
for epoch in range(epochs):
    for data, targets in loader_train:
        optimizer.zero_grad()            # 옵티마이저의 기울기 초기화
        y_pred = model(data)             # 순전파 연산으로 예측값 계산
        loss = loss_fn(y_pred, targets)  # 손실 함수로 비용 계산
        loss.backward()                  # 역전파 연산으로 기울기 계산
        optimizer.step()                 # 옵티마이저를 통해 파라미터 업데이트
    print('Epoch {:4d}/{} Cost: {:.6f}'.format(epoch + 1, 3, loss.item()))

## 테스트 데이터로 평가하기

`model.eval()` 로 모델을 추론 모드로 전환한다. 일부 계층은 학습과 추론 시 다르게 동작하기 때문이다. `torch.no_grad()` 로 기울기 계산을 비활성화해 메모리·속도를 개선한다. `torch.max(outputs.data, 1)` 로 각 예측에서 가장 높은 확률의 클래스(예측 레이블)를 찾아 실제 레이블과 비교한다.

In [ ]:
model.eval()  # 신경망을 추론 모드로 전환
correct = 0

# 데이터로더에서 미니배치를 하나씩 꺼내 추론을 수행
with torch.no_grad():  # 추론 과정에는 미분이 필요없음
    for data, targets in loader_test:
        outputs = model(data)  # 데이터를 입력하고 출력을 계산

        # 추론 계산
        _, predicted = torch.max(outputs.data, 1)  # 확률이 가장 높은 레이블이 무엇인지 계산
        correct += predicted.eq(targets.data.view_as(predicted)).sum()  # 정답과 일치한 경우 정답 카운트를 증가

# 정확도 출력
data_num = len(loader_test.dataset)  # 데이터 총 건수
print('\n테스트 데이터에서 예측 정확도: {}/{} ({:.0f}%)\n'.format(
    correct, data_num, 100. * correct / data_num))

약 96%의 정확도가 나온다. 마지막으로 특정 인덱스의 테스트 이미지 하나를 모델에 전달해 예측 결과를 확인하고 시각화한다.

In [ ]:
index = 2018

model.eval()  # 신경망을 추론 모드로 전환
data = X_test[index]
output = model(data)  # 데이터를 입력하고 출력을 계산
_, predicted = torch.max(output.data, 0)  # 확률이 가장 높은 레이블이 무엇인지 계산

print("예측 결과 : {}".format(predicted))

X_test_show = (X_test[index]).numpy()
plt.imshow(X_test_show.reshape(28, 28), cmap='gray')
print("이 이미지 데이터의 정답 레이블은 {:.0f}입니다".format(y_test[index]))

# 7. 과적합(Overfitting)을 막는 방법들

과적합은 모델 성능을 떨어뜨리는 주요 이슈다. 모델이 훈련 데이터를 불필요할 정도로 과하게 암기해(노이즈까지 학습) 새로운 데이터에서 제대로 동작하지 않는 상태다. 인공 신경망의 과적합을 막는 방법들을 정리한다.

## 1. 데이터의 양 늘리기

데이터가 적으면 모델이 특정 패턴이나 노이즈까지 쉽게 암기해 과적합이 발생하기 쉽다. 데이터를 늘릴수록 모델은 일반적인 패턴을 학습한다. 데이터가 적을 때는 기존 데이터를 조금씩 변형·추가해 양을 늘리는 **데이터 증강(Data Augmentation)**을 쓰기도 한다. 이미지의 경우 회전·노이즈 추가·일부 수정 등으로 증식시킨다.

## 2. 모델의 복잡도 줄이기

인공 신경망의 복잡도는 은닉층의 수나 매개변수의 수로 결정된다. 과적합이 포착되면 신경망의 복잡도를 줄일 수 있다. (신경망에서 매개변수의 수를 모델의 **수용력(capacity)**이라 부르기도 한다.)

예를 들어 아래처럼 3개의 선형 레이어를 가진 신경망이 과적합을 보인다면,

```python
class Architecture1(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(Architecture1, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc3 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.fc3(out)
        return out
```

다음처럼 레이어를 2개로 줄여 복잡도를 낮출 수 있다.

```python
class Architecture1(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(Architecture1, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out
```

## 3. 가중치 규제 (Regularization)

복잡한 모델은 간단한 모델보다 과적합되기 쉽다. 복잡한 모델을 간단하게 만드는 방법으로 **가중치 규제**가 있다.

- **L1 규제** : 가중치 $w$ 들의 **절대값 합**을 비용 함수에 추가한다(L1 노름). 기존 비용에 $\lambda |w|$ 를 더한다.
- **L2 규제** : 가중치 $w$ 들의 **제곱합**을 비용 함수에 추가한다(L2 노름). 기존 비용에 $\frac{1}{2}\lambda w^2$ 를 더한다.

여기서 $\lambda$ 는 규제의 강도를 정하는 하이퍼파라미터다. $\lambda$ 가 크면 모델이 훈련 데이터에 적합한 매개변수를 찾기보다 규제 항을 작게 유지하는 것을 우선한다.

두 식 모두 비용을 최소화하려면 가중치 $w$ 가 작아져야 한다. 예를 들어 $H(x) = w_1 x_1 + w_2 x_2 + w_3 x_3 + w_4 x_4$ 에 L1 규제를 적용했더니 $w_3$ 가 0이 되었다면, 이는 특성 $x_3$ 가 모델 결과에 별 영향을 주지 못함을 의미한다.

> **L1 vs L2** : L1은 어떤 특성이 모델에 영향을 주는지 정확히 판단하고자 할 때 유용하다(일부 $w$ 가 완전히 0이 됨). 그런 판단이 필요 없다면 경험적으로 L2가 더 잘 동작하므로 L2를 권장한다. 신경망에서 L2 규제는 **가중치 감쇠(weight decay)**라고도 부른다.

파이토치에서는 옵티마이저의 `weight_decay` 매개변수로 L2 규제를 적용한다(기본값 0).

In [ ]:
# (예시) 옵티마이저에 weight_decay를 설정하여 L2 규제를 적용한다.
# model = Architecture1(10, 20, 2)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

> 참고로 책에 따라 Regularization을 '정규화'로 번역하기도 하지만, 이는 Normalization(정규화)과 혼동되므로 **규제** 또는 정형화로 부르는 것이 바람직하다. 신경망에서 Normalization이라는 용어가 쓰이는 기법으로는 배치 정규화, 층 정규화 등이 따로 있다(8절에서 다룬다).

## 4. 드롭아웃 (Dropout)

**드롭아웃**은 학습 과정에서 신경망의 일부를 사용하지 않는 방법이다. 예를 들어 드롭아웃 비율을 0.5로 하면 학습 과정마다 랜덤으로 절반의 뉴런만 사용한다.

> 드롭아웃은 신경망 학습 시에만 사용하고, **예측(추론) 시에는 사용하지 않는** 것이 일반적이다. 학습 시 특정 뉴런이나 특정 조합에 너무 의존적이게 되는 것을 방지하고, 매번 다른 뉴런들을 사용하므로 서로 다른 신경망을 앙상블하는 것 같은 효과를 내어 과적합을 막는다.

(파이토치에서는 `nn.Dropout(p)` 레이어로 적용하며, `model.eval()` 로 추론 모드에 들어가면 드롭아웃이 자동으로 비활성화된다.)

# 8. 기울기 소실(Gradient Vanishing)과 폭주(Exploding)

깊은 신경망을 학습하다 보면 역전파 과정에서 입력층으로 갈수록 기울기가 점점 작아지는 현상이 생길 수 있다. 입력층에 가까운 가중치가 제대로 업데이트되지 않으면 최적 모델을 찾을 수 없다. 이를 **기울기 소실(Gradient Vanishing)**이라 한다.

반대로 기울기가 점점 커져 가중치가 비정상적으로 큰 값이 되며 발산하기도 한다. 이를 **기울기 폭주(Gradient Exploding)**라 하며, 뒤에서 배울 순환 신경망(RNN)에서 발생할 수 있다. 이 절에서는 이를 막는 방법들을 다룬다.

## 1. ReLU와 ReLU의 변형들

시그모이드 함수는 입력의 절대값이 크면 출력이 0 또는 1에 수렴하며 기울기가 0에 가까워진다. 그래서 역전파에서 전파할 기울기가 점차 사라져 기울기 소실이 발생한다.

기울기 소실을 완화하는 가장 간단한 방법은 **은닉층의 활성화 함수로 시그모이드·tanh 대신 ReLU나 Leaky ReLU 같은 ReLU 계열을 쓰는 것**이다.

> - 은닉층에서는 시그모이드 함수를 쓰지 않는다.
> - Leaky ReLU를 쓰면 모든 입력값에서 기울기가 0에 수렴하지 않아 죽은 ReLU 문제를 해결한다.
> - 은닉층에서는 ReLU나 Leaky ReLU 같은 ReLU 계열의 변형을 사용한다.

## 2. 가중치 초기화 (Weight Initialization)

같은 모델이라도 가중치 초기값에 따라 훈련 결과가 달라진다. 가중치 초기화만 잘해도 기울기 소실 같은 문제를 완화할 수 있다.

### 세이비어 초기화 (Xavier Initialization)

세이비어 글로럿과 요슈아 벤지오가 제안한 방법으로, 글로럿 초기화라고도 한다. 이전 층의 뉴런 수 $n_{in}$ 과 다음 층의 뉴런 수 $n_{out}$ 을 모두 사용한다.

- 균등 분포로 초기화할 경우:

$$ W \sim Uniform\left(-\sqrt{\frac{6}{n_{in} + n_{out}}},\ +\sqrt{\frac{6}{n_{in} + n_{out}}}\right) $$

- 정규 분포로 초기화할 경우(평균 0, 표준편차 $\sigma$):

$$ \sigma = \sqrt{\frac{2}{n_{in} + n_{out}}} $$

세이비어 초기화는 여러 층의 기울기 분산 사이에 균형을 맞춘다. 시그모이드·tanh 같은 S자 형태 활성화 함수와 함께 쓸 때 좋은 성능을 보이지만, **ReLU와 함께 쓰면 성능이 좋지 않다.**

### He 초기화 (He Initialization)

ReLU 계열과 함께 쓸 때 적합한 방법이다. 세이비어와 유사하나 **다음 층의 뉴런 수를 반영하지 않고** $n_{in}$ 만 사용한다.

- 균등 분포로 초기화할 경우:

$$ W \sim Uniform\left(-\sqrt{\frac{6}{n_{in}}},\ +\sqrt{\frac{6}{n_{in}}}\right) $$

- 정규 분포로 초기화할 경우:

$$ \sigma = \sqrt{\frac{2}{n_{in}}} $$

> **정리** : 시그모이드·tanh를 쓰면 세이비어 초기화, ReLU 계열을 쓰면 He 초기화가 효율적이다. **ReLU + He 초기화** 조합이 좀 더 보편적이다.

## 3. 배치 정규화 (Batch Normalization)

ReLU 계열과 He 초기화만으로도 기울기 소실/폭주를 어느 정도 완화할 수 있지만, 훈련 중 언제든 다시 발생할 수 있다. 또 다른 예방법이 **배치 정규화**다. 배치 정규화는 인공 신경망의 각 층에 들어가는 입력을 평균과 분산으로 정규화하여 학습을 효율적으로 만든다.

### 내부 공변량 변화 (Internal Covariate Shift)

- **공변량 변화** : 훈련 데이터의 분포와 테스트 데이터의 분포가 다른 경우.
- **내부 공변량 변화** : 학습 과정에서 **층 사이에서** 입력 데이터의 분포가 달라지는 현상. 이전 층의 가중치가 바뀌면 현재 층에 전달되는 입력 분포가 달라진다. 배치 정규화 논문은 딥 러닝의 불안정성이 층마다 입력 분포가 달라지기 때문이라고 주장한다.

### 배치 정규화의 수식

배치 정규화는 한 번에 들어오는 배치 단위로 정규화하며, **각 층에서 활성화 함수를 통과하기 전에** 수행된다. 입력에 대해 평균을 0으로 만들고 정규화한 뒤, 스케일($\gamma$)과 시프트($\beta$)를 적용한다($\gamma, \beta$ 는 학습 대상).

미니 배치 $B = x^{(1)}, x^{(2)}, \dots, x^{(m)}$ 에 대해,

$$ \mu_B \leftarrow \frac{1}{m}\sum_{i=1}^{m} x^{(i)} \quad \text{(미니 배치에 대한 평균)} $$
$$ \sigma_B^2 \leftarrow \frac{1}{m}\sum_{i=1}^{m}(x^{(i)} - \mu_B)^2 \quad \text{(미니 배치에 대한 분산)} $$
$$ \hat{x}^{(i)} \leftarrow \frac{x^{(i)} - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \quad \text{(정규화)} $$
$$ y^{(i)} \leftarrow \gamma \hat{x}^{(i)} + \beta = BN_{\gamma, \beta}(x^{(i)}) \quad \text{(스케일 조정과 시프트)} $$

여기서 $\epsilon$ 은 분모가 0이 되는 것을 막는 작은 수다(보편적으로 $10^{-5}$). 학습 시에는 배치마다 평균·분산을 구해 이동 평균·이동 분산을 저장해 두었다가, 테스트 시에는 저장해 둔 값으로 정규화한다.

### 배치 정규화의 효과와 한계

**효과**

- 시그모이드·tanh를 써도 기울기 소실 문제가 크게 개선된다.
- 가중치 초기화에 훨씬 덜 민감해진다.
- 훨씬 큰 학습률을 쓸 수 있어 학습 속도가 개선된다.
- 미니 배치마다 평균·표준편차를 계산하므로 훈련 데이터에 잡음을 넣는 부수 효과로 과적합을 방지하기도 한다(단 부수 효과이므로 드롭아웃과 함께 쓰는 것이 좋다).

**한계**

- **미니 배치 크기에 의존적이다.** 배치 크기가 너무 작으면 잘 동작하지 않는다(배치 크기 1이면 분산이 0). 어느 정도 크기가 되는 미니 배치에서 쓰는 것이 좋다.
- **RNN에 적용하기 어렵다.** RNN은 시점(time step)마다 다른 통계치를 가지기 때문이다. 또한 모델을 복잡하게 하고 추가 계산을 하므로 테스트 시 실행 시간이 느려진다.

## 4. 층 정규화 (Layer Normalization)

배치 정규화의 한계를 보완하는 방법으로, 배치 크기에 의존적이지 않고 RNN에도 적용하기 수월하다.

두 정규화의 차이를 직관적으로 정리하면 다음과 같다(미니 배치는 동일한 특성 개수를 가진 다수의 샘플들을 의미함을 상기한다).

- **배치 정규화** : **같은 특성(feature)별로**, 미니 배치 내 여러 샘플에 걸쳐 평균·분산을 구해 정규화한다(열 방향).
- **층 정규화** : **같은 샘플 내에서**, 그 샘플의 모든 특성에 걸쳐 평균·분산을 구해 정규화한다(행 방향).

따라서 층 정규화는 각 샘플을 독립적으로 정규화하므로 배치 크기에 무관하게 동작한다.

## (실습) 파이토치에서 가중치 초기화와 배치 정규화 적용하기

위 개념을 코드로 확인한다. ReLU 은닉층에 **He 초기화**를 적용하고, 활성화 함수 직전에 **배치 정규화**(`nn.BatchNorm1d`)를 넣은 간단한 MLP를 정의해 본다. (구조 확인용이며, 실제 학습 루프는 6절과 동일한 방식으로 붙이면 된다.)

In [ ]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(784, 100),
    nn.BatchNorm1d(100),  # 활성화 함수 통과 전에 배치 정규화
    nn.ReLU(),
    nn.Linear(100, 100),
    nn.BatchNorm1d(100),
    nn.ReLU(),
    nn.Linear(100, 10)
)

# ReLU 계열에 적합한 He 초기화를 선형 레이어에 적용
for layer in model:
    if isinstance(layer, nn.Linear):
        nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')  # He(=Kaiming) 초기화
        nn.init.zeros_(layer.bias)

print(model)

> `nn.init.kaiming_uniform_` 가 곧 He 초기화의 균등 분포 버전이다(He = Kaiming He). `nonlinearity='relu'` 로 ReLU에 맞는 스케일을 적용한다. 배치 정규화 레이어가 들어가면 `model.train()` / `model.eval()` 전환이 더 중요해진다 — 학습 시에는 배치 통계를, 추론 시에는 저장해 둔 이동 평균·이동 분산을 사용하기 때문이다.

---

이로써 6장 **인공 신경망**의 핵심을 모두 정리했다. 퍼셉트론에서 출발해 다층 퍼셉트론으로 XOR을 풀고, 역전파로 가중치가 학습되는 원리를 직접 계산으로 확인했으며, 손글씨·MNIST 분류 실습과 함께 과적합·기울기 소실을 막는 실전 기법까지 살펴봤다. 다음 장에서는 시퀀스 데이터를 다루는 **순환 신경망(RNN)**으로 넘어간다.